# 📰 네이버 뉴스 스크래핑: Network API 활용 (VSCode 실습용)

---

이 노트북은 **브라우저 개발자 도구(Network 탭)**를 활용하여
네이버 뉴스 검색 결과를 **스크래핑**하는 방법을 단계별로 학습합니다.

HTML 구조를 읽고 CSS 선택자를 작성하여 원하는 데이터를 추출하는 **스크래핑 코드 기초**를 익히는 것이 목표입니다.

### 학습 로드맵

| 순서 | 주제 | 핵심 내용 |
|------|------|-----------|
| 0 | 웹 스크래핑이란? | 개념, 동작 원리, 윤리적 고려사항 |
| 1 | VSCode 환경 설정 | 가상환경, 패키지 설치 |
| 2 | Network API 방식 이해 | 개발자 도구 활용, 요청 URL 분석 |
| 3 | 라이브러리 불러오기 | requests, BeautifulSoup, pandas |
| 4 | HTML 구조 분석 | 태그, 클래스, CSS 선택자 |
| 5 | 단일 페이지 스크래핑 | 요청 → 파싱 → 데이터 추출 |
| 6 | 언론사 ID 매핑 | 네이버 뉴스 언론사 코드 |
| 7 | 스크래핑 함수 완성 | 다중 페이지, 매개변수 처리 |
| 8 | 실행 및 결과 저장 | 검색 실행, Excel 저장 |
| - | Quick Reference | 핵심 문법 요약표 |

## 0. 웹 스크래핑이란?

### 개념

**웹 스크래핑(Web Scraping)**은 웹 페이지에서 **원하는 데이터를 자동으로 추출**하는 기술입니다.

```
브라우저가 하는 일을 Python 코드로 대신함:
    요청(Request) → 응답(Response) → HTML 파싱 → 데이터 추출
```

### 두 가지 방식

| 방식 | 설명 | 장점 | 단점 |
|------|------|------|------|
| **HTML 직접 파싱** | 검색 결과 페이지의 HTML을 직접 요청 | 간단, 빠름 | 네이버 구조 변경 시 깨짐 |
| **Network API 활용** | 브라우저가 내부적으로 호출하는 API를 찾아 직접 요청 | JSON 응답, 안정적 | API URL 분석 필요 |

> **이 노트북에서는 Network API 방식**을 사용합니다.
> 브라우저 개발자 도구에서 실제 데이터를 전달하는 API 요청을 찾아 활용합니다.

### 윤리적 고려사항

| 항목 | 내용 |
|------|------|
| `robots.txt` | 사이트의 크롤링 허용 범위 확인 (예: `https://search.naver.com/robots.txt`) |
| 요청 간격 | 서버에 부담을 주지 않도록 `time.sleep()` 으로 **최소 1초 이상** 대기 |
| 이용 약관 | 상업적 목적의 대량 수집은 법적 문제가 될 수 있음 |
| 개인정보 | 개인정보가 포함된 데이터 수집·저장에 주의 |

## 1. VSCode 환경 설정 (가상환경 및 패키지 설치)

### 1단계: 가상환경 생성 (터미널에서 실행)

```bash
# 프로젝트 폴더로 이동
cd my_project

# 가상환경 생성
python -m venv .venv

# 가상환경 활성화
# Windows:
.venv\\Scripts\\activate
# macOS/Linux:
source .venv/bin/activate
```

### 2단계: 패키지 설치

```bash
pip install requests beautifulsoup4 pandas openpyxl tqdm ipykernel
```

| 패키지 | 용도 |
|--------|------|
| `requests` | HTTP 요청 (웹 페이지/API 호출) |
| `beautifulsoup4` | HTML 파싱 (태그에서 데이터 추출) |
| `pandas` | 데이터프레임 (표 형태 정리) |
| `openpyxl` | Excel 파일 저장 |
| `tqdm` | 진행률 표시 바 |
| `ipykernel` | Jupyter가 가상환경 Python을 커널로 인식 |

> 💡 **ipykernel**은 Jupyter가 가상환경의 Python을 커널로 인식하기 위해 반드시 필요합니다.

### 3단계: VSCode 커널 선택

1. `.ipynb` 파일 열기
2. 우측 상단 **커널 선택** 클릭
3. `.venv (Python 3.x.x)` 선택

## 2. Network API 방식 이해하기

### 브라우저 개발자 도구(DevTools)로 API 찾는 방법

#### 1단계: 네이버 뉴스 검색 페이지 열기

1. 브라우저에서 `https://search.naver.com/search.naver?where=news&query=파이썬` 접속
2. `F12` 또는 `Ctrl+Shift+I` 로 **개발자 도구** 열기
3. **Network** 탭 클릭

#### 2단계: 스크롤 다운하며 API 요청 찾기

1. Network 탭이 열린 상태에서 **검색 결과 페이지를 아래로 스크롤**
2. 스크롤할 때마다 `more?cluster...` 또는 `more?...` 형태의 **XHR 요청**이 생김
3. 이 요청을 클릭하면:
   - **Headers** 탭 → 요청 URL, 쿼리 파라미터, User-Agent 등 확인
   - **Response** 탭 → JSON 형태의 응답 데이터 (뉴스 10건씩 포함)

#### 3단계: 요청 URL 구조 분석

```
https://s.search.naver.com/p/newssearch/3/api/tab/more
  ?query=파이썬              ← 검색어 (URL 인코딩)
  &ssc=tab.news.all          ← 검색 범위
  &pd=3                      ← 기간 검색 모드
  &photo=0                   ← 뉴스 유형 (0=전체)
  &sort=0                    ← 정렬 (0=관련도, 1=최신, 2=오래된순)
  &ds=2025.01.01             ← 시작 날짜
  &de=2025.03.01             ← 종료 날짜
  &start=1                   ← 시작 위치 (1, 11, 21, ...)
  &news_office_checked=1056  ← 언론사 ID (비우면 전체)
  &mynews=1                  ← 언론사 필터 활성화
  &nso=...                   ← 기간 필터 인코딩값
```

> **핵심 포인트**: `start` 값을 1, 11, 21, ... 으로 증가시키면
> 한 페이지(10건)씩 다음 결과를 받아올 수 있습니다.

### 응답 구조 (JSON)

```json
{
  "collection": [
    {
      "html": "<div class=\"sds-comps-...\">..(뉴스 10건 HTML)..</div>"
    }
  ]
}
```

- API 응답은 JSON이며, `collection[0].html`에 뉴스 HTML이 담겨 있음
- 이 HTML을 BeautifulSoup으로 파싱하여 개별 기사 정보를 추출

## 3. 라이브러리 불러오기

In [ ]:
# ──────────────────────────────────────────────
# 라이브러리 불러오기 + 환경 확인
# ──────────────────────────────────────────────
import sys
print(f"Python 버전: {sys.version}")
print(f"Python 경로: {sys.executable}")
print()

import requests              # HTTP 요청 라이브러리
from bs4 import BeautifulSoup  # HTML 파싱 라이브러리
import urllib.parse          # URL 인코딩 (한글 → %ED%8C%8C%EC%9D%B4%EC%8D%AC)
import pandas as pd          # 데이터프레임
import time                  # 요청 간 대기 시간
from tqdm import tqdm        # 진행률 표시 바

print(f"requests 버전: {requests.__version__}")
print(f"pandas 버전:   {pd.__version__}")
print("🎉 라이브러리 로드 완료!")

## 4. HTML 구조 분석과 CSS 선택자

### HTML 기본 구조

웹 페이지는 **HTML 태그**로 구성됩니다. 스크래핑은 이 태그에서 원하는 데이터를 찾는 작업입니다.

```html
<div class="news_wrap">           ← div 태그, class 속성
  <a class="news_tit" href="URL"> ← a 태그 (링크), href 속성
    <span>기사 제목</span>         ← span 태그 안의 텍스트
  </a>
  <div class="news_dsc">
    기사 본문 요약...
  </div>
</div>
```

### CSS 선택자 (CSS Selector) 기본 문법

BeautifulSoup의 `select()` 메서드에서 사용하는 선택자입니다.

| 선택자 | 의미 | 예시 |
|--------|------|------|
| `태그명` | 해당 태그 모두 선택 | `div`, `a`, `span` |
| `.클래스명` | 해당 클래스를 가진 요소 | `.news_tit` |
| `#아이디` | 해당 ID를 가진 요소 | `#main` |
| `태그.클래스` | 태그+클래스 조합 | `a.news_tit` |
| `부모 자식` | 부모 안의 자식 요소 | `div.news_wrap a` |
| `부모 > 자식` | 직접 자식만 | `div > a` |

### 네이버 뉴스 API 응답의 HTML 구조

Network API 응답으로 받은 HTML의 구조를 분석하면:

```
div.sds-comps-vertical-layout...  ← 각 뉴스 기사 1건
├── span.sds-comps-profile-info-title-text    ← 언론사 영역
│   └── span.sds-comps-text                   ← 언론사명
├── span.sds-comps-profile-info-subtext       ← 날짜/부가정보 영역
│   └── span.sds-comps-text                   ← 날짜 텍스트
├── a.T2s2rixtehFC2DsDea0v (href에 naver.com) ← 네이버뉴스 링크
├── a.T2s2rixtehFC2DsDea0v.jKya96OjY1RWvTVfCpBm ← 언론사 원문 링크
├── span.sds-comps-text-type-headline1        ← 기사 제목
└── span.sds-comps-text-ellipsis-3...body1    ← 기사 본문 요약 (3줄)
```

> **참고**: 네이버는 주기적으로 HTML 구조(클래스명 등)를 변경합니다.
> 스크래핑이 안 되면 개발자 도구로 현재 클래스명을 다시 확인하세요.

## 5. 단일 페이지 스크래핑 (단계별)

실제 Network API를 호출하여 **1페이지(10건)**를 가져오는 과정을 단계별로 실습합니다.

In [ ]:
# ──────────────────────────────────────────────
# 단계 1: Network API 요청 보내기
#   브라우저가 내부적으로 호출하는 API를 직접 호출
#   query, 날짜, 정렬 등을 URL 파라미터로 전달
# ──────────────────────────────────────────────

# ---- 검색 조건 설정 ----
query = "파이썬"                 # 검색어
date_from = "20250101"           # 시작 날짜 (YYYYMMDD)
date_to   = "20250301"           # 종료 날짜 (YYYYMMDD)
sort = "1"                       # 0=관련도순, 1=최신순, 2=오래된순
start = "1"                      # 시작 위치 (1=첫 페이지)

# ---- URL 구성 ----
q = urllib.parse.quote(query)    # 한글 → URL 인코딩
ds = f"{date_from[:4]}.{date_from[4:6]}.{date_from[6:]}"   # "2025.01.01"
de = f"{date_to[:4]}.{date_to[4:6]}.{date_to[6:]}"         # "2025.03.01"
nso = f"so%3Ar%2Cp%3Afrom{date_from}to{date_to}%2Ca%3Aall" # 기간 필터 인코딩

target_url = (
    "https://s.search.naver.com/p/newssearch/3/api/tab/more"
    f"?query={q}"
    f"&ssc=tab.news.all"
    f"&pd=3"
    f"&photo=0"
    f"&sort={sort}"
    f"&nso={nso}"
    f"&ds={ds}&de={de}"
    f"&start={start}"
    f"&news_office_checked="
    f"&mynews=0"
)

print("요청 URL:")
print(target_url[:120] + "...")
print()

# ---- HTTP 요청 ----
#   User-Agent: 브라우저처럼 보이기 위한 헤더 (차단 방지)
#   Referer: 네이버 검색 페이지에서 온 것처럼 설정
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/135.0.0.0 Safari/537.36",
    "Referer": f"https://search.naver.com/search.naver?where=news&query={q}"
}

response = requests.get(target_url, headers=headers)

print(f"응답 상태 코드: {response.status_code}")   # 200이면 성공
print(f"응답 크기: {len(response.text):,} 글자")

In [ ]:
# ──────────────────────────────────────────────
# 단계 2: JSON 응답에서 HTML 추출
#   API 응답은 JSON 형태
#   collection[0].html 에 뉴스 HTML이 담겨 있음
# ──────────────────────────────────────────────

data = response.json()

# JSON 구조 확인
print("응답 JSON 키:", list(data.keys()))
print(f"collection 개수: {len(data.get('collection', []))}")
print()

# HTML 추출
html = data['collection'][0].get('html', '')
print(f"HTML 길이: {len(html):,} 글자")
print()

# HTML 미리보기 (앞 500자)
print("HTML 미리보기:")
print(html[:500])
print("...")

In [ ]:
# ──────────────────────────────────────────────
# 단계 3: BeautifulSoup으로 HTML 파싱
#   HTML 문자열 → BeautifulSoup 객체로 변환
#   CSS 선택자로 각 뉴스 기사 블록 선택
# ──────────────────────────────────────────────

soup = BeautifulSoup(html, 'html.parser')

# 각 기사 아이템 선택
#   네이버 뉴스 API 응답에서 각 기사는 아래 클래스를 가진 div
news_items = soup.select(
    'div.sds-comps-vertical-layout.sds-comps-full-layout.OyoX13laVzzftRqUMHQP'
)

print(f"찾은 뉴스 기사 수: {len(news_items)}건")
print()

# 첫 번째 기사의 HTML 구조 살펴보기
if news_items:
    first = news_items[0]
    print("=== 첫 번째 기사 HTML (일부) ===")
    print(str(first)[:800])
    print("...")

In [ ]:
# ──────────────────────────────────────────────
# 단계 4: 각 기사에서 개별 데이터 추출
#   CSS 선택자로 언론사, 날짜, 제목, 본문, URL 추출
#   select_one() = 첫 번째 일치 요소 1개 반환
#   select()     = 일치하는 모든 요소 리스트 반환
# ──────────────────────────────────────────────

results = []

for news_item in news_items:
    # ---- 언론사명 ----
    media_elem = news_item.select_one(
        'span.sds-comps-profile-info-title-text span.sds-comps-text'
    )
    media_name = media_elem.text.strip() if media_elem else ''

    # ---- 날짜 ----
    date_text = ''
    date_spans = news_item.select(
        'span.sds-comps-profile-info-subtext '
        'div.DJwZySR1gWTQoLm3xvvD span.sds-comps-text'
    )
    for span in date_spans:
        text = span.text.strip()
        # "YYYY.MM.DD" 형식인지 확인 (마침표가 2개 이상)
        if '.' in text and len(text.split('.')) >= 3:
            if not any(kw in text for kw in ['면', '단', '네이버']):
                date_text = text
                break

    # ---- 네이버뉴스 URL ----
    naver_url = ''
    naver_links = news_item.select('a.T2s2rixtehFC2DsDea0v.DJwZySR1gWTQoLm3xvvD')
    for link in naver_links:
        if link.get('href') and 'naver.com' in link.get('href', ''):
            if '네이버' in link.get_text(strip=True):
                naver_url = link['href']
                break

    # ---- 언론사 원문 URL ----
    media_url = ''
    media_links = news_item.select('a.T2s2rixtehFC2DsDea0v.jKya96OjY1RWvTVfCpBm')
    if media_links:
        media_url = media_links[0].get('href', '')

    # ---- 제목 ----
    title_elem = news_item.select_one('span.sds-comps-text-type-headline1')
    title = title_elem.text.strip() if title_elem else ''

    # ---- 본문 요약 ----
    text_elem = news_item.select_one(
        'span.sds-comps-text-ellipsis-3.sds-comps-text-type-body1'
    )
    text = text_elem.get_text(strip=True) if text_elem else ''

    # 결과 리스트에 추가
    if media_name or title:
        results.append([media_name, date_text, title, text, naver_url, media_url])

# 결과 출력
print(f"추출된 기사 수: {len(results)}건\n")
for i, r in enumerate(results[:3]):  # 처음 3건만 미리보기
    print(f"--- 기사 {i+1} ---")
    print(f"  언론사: {r[0]}")
    print(f"  날짜:   {r[1]}")
    print(f"  제목:   {r[2][:50]}...")
    print(f"  본문:   {r[3][:50]}...")
    print()

In [ ]:
# ──────────────────────────────────────────────
# 단계 5: DataFrame으로 정리
#   추출한 결과를 pandas DataFrame으로 변환
# ──────────────────────────────────────────────

df = pd.DataFrame(
    results,
    columns=['media', 'date', 'title', 'text', 'naver_url', 'media_url']
)

print(f"총 {len(df)}건 수집됨\n")
df.head()

## 6. 언론사별 네이버 ID 매핑

네이버 뉴스 검색에서 **특정 언론사**만 필터링하려면, 해당 언론사의 **네이버 고유 ID**를 알아야 합니다.
URL 파라미터 `news_office_checked=ID` 로 전달합니다.

> 아래 딕셔너리는 주요 언론사의 네이버 ID를 정리한 것입니다.
> 네이버 뉴스 검색 → 언론사 필터 → 개발자 도구에서 ID를 확인할 수 있습니다.

In [ ]:
# ──────────────────────────────────────────────
# 언론사별 네이버 ID 딕셔너리
#   news_office_checked 파라미터에 사용
#   '전체'를 지정하면 모든 언론사 검색
# ──────────────────────────────────────────────

news_office = {
    # ── 종합 일간지 ──
    '경향신문': 1032,  '국민일보': 1005,  '동아일보': 1020,
    '문화일보': 1021,  '서울신문': 1081,  '세계일보': 1022,
    '조선일보': 1023,  '중앙일보': 1025,  '한겨레': 1028,
    '한국일보': 1469,

    # ── 통신사 ──
    '뉴스1': 1421,     '뉴시스': 1003,    '연합뉴스': 1001,
    '연합뉴스TV': 1422,

    # ── 방송사 ──
    'KBS': 1056,       'MBC': 1214,       'SBS': 1055,
    'JTBC': 1437,      'MBN': 1019,       'YTN': 1052,
    '채널A': 1449,     'TV조선': 1448,    'SBS Biz': 1374,
    '한국경제TV': 1215,

    # ── 경제지 ──
    '매일경제': 1009,  '머니투데이': 1008, '서울경제': 1011,
    '아시아경제': 1277, '이데일리': 1018,  '조선비즈': 1366,
    '파이낸셜뉴스': 1014, '한국경제': 1015, '헤럴드경제': 1016,
    '비즈워치': 1648,  '조세일보': 1123,

    # ── 인터넷 / IT 전문지 ──
    '노컷뉴스': 1079,  '더팩트': 1629,    '데일리안': 1119,
    '머니S': 1417,     '미디어오늘': 1006, '아이뉴스24': 1031,
    '오마이뉴스': 1047, '프레시안': 1002,  '디지털데일리': 1138,
    '디지털타임스': 1029, '블로터': 1293,  '전자신문': 1030,
    '지디넷코리아': 1092,

    # ── 주간/월간지 ──
    '더스쿠프': 1665,   '레이디경향': 1145, '매경이코노미': 1024,
    '시사IN': 1308,     '시사저널': 1586,   '신동아': 1262,
    '주간경향': 1033,   '주간동아': 1037,   '주간조선': 1053,
    '중앙SUNDAY': 1353, '한겨레21': 1036,   '한경비즈니스': 1050,

    # ── 전문/지역 ──
    '동아사이언스': 1584, '여성신문': 1310, '코메디닷컴': 1296,
    '헬스조선': 1346,    '강원일보': 1087,  '국제신문': 1658,
    '매일신문': 1088,    '부산일보': 1082,

    # ── 해외 통신 ──
    '신화사 연합뉴스': 1348, 'AP연합뉴스': 1077, 'EPA연합뉴스': 1091,
}

print(f"등록된 언론사 수: {len(news_office)}개")
print(f"\n예시: KBS ID = {news_office['KBS']}, 조선일보 ID = {news_office['조선일보']}")

## 7. 스크래핑 함수 완성

지금까지 단계별로 실습한 내용을 **하나의 함수**로 통합합니다.

### 함수 설계

```python
navernews_network_scrape(
    query      = "검색어",
    media      = "KBS" 또는 "전체",
    date_from  = 20250101,       # YYYYMMDD
    date_to    = 20250301,       # YYYYMMDD
    sort       = 1,              # 0=관련도, 1=최신, 2=오래된순
    news_type  = 0,              # 0=전체, 1=포토, 2=동영상, 3=지면, 4=보도자료
    pages      = 5               # 수집할 페이지 수 (1페이지 = 10건)
) → pd.DataFrame
```

### 매개변수 상세

| 매개변수 | 타입 | 기본값 | 설명 |
|----------|------|--------|------|
| `query` | str | '파이썬' | 검색 키워드 |
| `media` | str | '전체' | 언론사 이름 (news_office 키) 또는 '전체' |
| `date_from` | int | 20250101 | 검색 시작 날짜 (YYYYMMDD) |
| `date_to` | int | 20250301 | 검색 종료 날짜 (YYYYMMDD) |
| `sort` | int | 1 | 정렬 (0=관련도, 1=최신, 2=오래된순) |
| `news_type` | int | 0 | 뉴스 유형 필터 |
| `pages` | int | 1 | 수집 페이지 수 (최대 400페이지 = 4,000건) |

In [ ]:
# ──────────────────────────────────────────────
# 네이버 뉴스 Network API 스크래핑 함수
#   다중 페이지 자동 수집 + DataFrame 반환
# ──────────────────────────────────────────────

def navernews_network_scrape(
    query:     str = '파이썬',
    media:     str = '전체',       # news_office 키 또는 '전체'
    date_from: int = 20250101,     # YYYYMMDD
    date_to:   int = 20250301,     # YYYYMMDD
    sort:      int = 1,            # 0=관련도순, 1=최신순, 2=오래된순
    news_type: int = 0,            # 0=전체, 1=포토, 2=동영상, 3=지면, 4=보도자료, 5=자동생성
    pages:     int = 1,            # 수집할 페이지 수 (1페이지 = 10건)
) -> pd.DataFrame:
    """
    네이버 뉴스 검색 Network API를 호출하여 기사를 수집합니다.

    Returns:
        pd.DataFrame: columns=['media','date','title','text','naver_url','media_url']
    """

    # ---- 매개변수 변환 ----
    date_from_str = str(date_from)
    date_to_str   = str(date_to)

    q = urllib.parse.quote(query)   # URL 인코딩

    # 언론사 ID 결정
    if media == '전체':
        media_id = ''
        mynews = '0'                # 전체 검색
    else:
        media_id = news_office.get(media, '')
        mynews = '1'                # 언론사 필터 활성화

    # 날짜 포맷
    ds = f"{date_from_str[:4]}.{date_from_str[4:6]}.{date_from_str[6:]}"
    de = f"{date_to_str[:4]}.{date_to_str[4:6]}.{date_to_str[6:]}"
    nso = f"so%3Ar%2Cp%3Afrom{date_from_str}to{date_to_str}%2Ca%3Aall"

    # 요청 헤더
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/135.0.0.0 Safari/537.36"
        ),
        "Referer": f"https://search.naver.com/search.naver?where=news&query={q}"
    }

    # ---- 페이지별 수집 ----
    results_list = []

    for page in tqdm(range(1, pages + 1), desc="수집 중"):
        start = str((page - 1) * 10 + 1)   # 1, 11, 21, ...

        target_url = (
            "https://s.search.naver.com/p/newssearch/3/api/tab/more"
            f"?query={q}"
            f"&ssc=tab.news.all"
            f"&pd=3"
            f"&photo={news_type}"
            f"&sort={sort}"
            f"&nso={nso}"
            f"&ds={ds}&de={de}"
            f"&start={start}"
            f"&news_office_checked={media_id}"
            f"&mynews={mynews}"
            f"&office_type=1"
            f"&office_section_code=1"
        )

        response = requests.get(target_url, headers=headers)

        if response.status_code != 200:
            print(f"페이지 {page} 요청 실패: 상태 코드 {response.status_code}")
            continue

        try:
            data = response.json()

            # 응답 데이터 확인
            if not data.get('collection'):
                print(f"페이지 {page}: 수집할 데이터 없음")
                continue

            html = data['collection'][0].get('html', '')
            if not html:
                print(f"페이지 {page}: HTML 데이터 없음")
                continue

            soup = BeautifulSoup(html, 'html.parser')

            # 각 기사 블록 선택
            news_items = soup.select(
                'div.sds-comps-vertical-layout'
                '.sds-comps-full-layout'
                '.OyoX13laVzzftRqUMHQP'
            )

            if not news_items:
                print(f"페이지 {page}: 결과 없음. 수집 종료.")
                break

            # ---- 기사별 데이터 추출 ----
            for news_item in news_items:
                try:
                    # 언론사
                    elem = news_item.select_one(
                        'span.sds-comps-profile-info-title-text span.sds-comps-text'
                    )
                    media_name = elem.text.strip() if elem else ''

                    # 날짜
                    date_text = ''
                    for span in news_item.select(
                        'span.sds-comps-profile-info-subtext '
                        'div.DJwZySR1gWTQoLm3xvvD span.sds-comps-text'
                    ):
                        t = span.text.strip()
                        if '.' in t and len(t.split('.')) >= 3:
                            if not any(kw in t for kw in ['면', '단', '네이버']):
                                date_text = t
                                break

                    # 네이버뉴스 URL
                    naver_url = ''
                    for link in news_item.select(
                        'a.T2s2rixtehFC2DsDea0v.DJwZySR1gWTQoLm3xvvD'
                    ):
                        href = link.get('href', '')
                        if 'naver.com' in href and '네이버' in link.get_text(strip=True):
                            naver_url = href
                            break

                    # 언론사 원문 URL
                    media_url = ''
                    ml = news_item.select(
                        'a.T2s2rixtehFC2DsDea0v.jKya96OjY1RWvTVfCpBm'
                    )
                    if ml:
                        media_url = ml[0].get('href', '')

                    # 제목
                    te = news_item.select_one('span.sds-comps-text-type-headline1')
                    title = te.text.strip() if te else ''

                    # 본문 요약
                    be = news_item.select_one(
                        'span.sds-comps-text-ellipsis-3.sds-comps-text-type-body1'
                    )
                    body = be.get_text(strip=True) if be else ''

                    if media_name or title:
                        results_list.append([
                            media_name, date_text, title, body,
                            naver_url, media_url
                        ])

                except Exception as e:
                    print(f"기사 처리 오류: {e}")
                    continue

        except Exception as e:
            print(f"페이지 {page} 처리 오류: {e}")

        # 서버 부담 방지를 위한 대기
        time.sleep(1)

    # ---- 결과 DataFrame 생성 ----
    if results_list:
        df = pd.DataFrame(
            results_list,
            columns=['media', 'date', 'title', 'text', 'naver_url', 'media_url']
        )
        last_date = df['date'].iloc[-1] if not df.empty and df['date'].iloc[-1] else '날짜 정보 없음'
        print(f"\n{last_date}까지 총 {len(df)}건 수집 완료.")
        return df
    else:
        print("수집된 데이터가 없습니다.")
        return pd.DataFrame(
            columns=['media', 'date', 'title', 'text', 'naver_url', 'media_url']
        )

print("navernews_network_scrape() 함수 정의 완료")

## 8. 실행 및 결과 저장

함수를 실행하여 뉴스를 수집하고, Excel 파일로 저장합니다.

### 수집 팁

| 항목 | 설명 |
|------|------|
| 검색건수 확인 불가 | Network API에서는 총 건수를 알 수 없음 |
| 오래된 순 권장 | `sort=2`로 오래된 순 지정하면 날짜 기준 순차 수집 가능 |
| 최대 4,000건 | 1페이지 10건 × 최대 400페이지 |
| 4,000건 초과 시 | 마지막 수집 날짜를 확인 → 그 날짜부터 다음 수집 시작 |
| 요청 간격 | 1초 대기 (함수 내부에 `time.sleep(1)` 포함) |

In [ ]:
# ──────────────────────────────────────────────
# 뉴스 수집 실행
#   검색 조건을 설정하고 함수 호출
# ──────────────────────────────────────────────

# ---- 검색 조건 설정 ----
query     = '경희대'          # 검색어
media     = '전체'            # 언론사 ('전체' 또는 news_office 키)
date_from = 20250501          # 시작 날짜 (YYYYMMDD)
date_to   = 20250527          # 종료 날짜 (YYYYMMDD)
sort      = 2                 # 0=관련도순, 1=최신순, 2=오래된순
news_type = 0                 # 0=전체, 1=포토, 2=동영상
pages     = 2                 # 수집 페이지 수 (2페이지 = 최대 20건)

# ---- 함수 실행 ----
final_result = navernews_network_scrape(
    query=query,
    media=media,
    date_from=date_from,
    date_to=date_to,
    sort=sort,
    news_type=news_type,
    pages=pages
)

# ---- 마지막 수집 날짜 확인 ----
#   4,000건 초과 시, 이 날짜부터 다음 수집을 시작하세요
if not final_result.empty:
    last_date = final_result['date'].iloc[-1]
    print(f"\n마지막 수집 기사 날짜: {last_date}")
else:
    print("수집된 데이터가 없습니다.")

# ---- 결과 미리보기 ----
final_result.head()

In [ ]:
# ──────────────────────────────────────────────
# 결과 저장 (Excel)
#   openpyxl 패키지 필요: pip install openpyxl
# ──────────────────────────────────────────────

# 파일명에 검색어와 날짜 포함
filename = f"{query}_news_{date_from}_{date_to}.xlsx"

final_result.to_excel(filename, index=False)
print(f"저장 완료: {filename}")
print(f"총 {len(final_result)}건")

---

## 네이버 뉴스 스크래핑 핵심 요약 (Quick Reference)

### 주요 라이브러리

| 라이브러리 | 용도 | 핵심 함수/메서드 |
|-----------|------|------------------|
| `requests` | HTTP 요청 | `requests.get(url, headers=...)` |
| `BeautifulSoup` | HTML 파싱 | `soup.select()`, `soup.select_one()` |
| `urllib.parse` | URL 인코딩 | `urllib.parse.quote("한글")` |
| `pandas` | 데이터 정리/저장 | `pd.DataFrame()`, `df.to_excel()` |

### CSS 선택자 문법

| 선택자 | 의미 | 예시 |
|--------|------|------|
| `태그명` | 태그 선택 | `div`, `span`, `a` |
| `.클래스` | 클래스 선택 | `.news_tit` |
| `태그.클래스` | 태그+클래스 | `span.sds-comps-text` |
| `부모 자식` | 하위 요소 | `div.news_wrap a.news_tit` |
| `[속성=값]` | 속성 선택 | `a[href*="naver.com"]` |

### BeautifulSoup 핵심 메서드

| 메서드 | 반환 | 설명 |
|--------|------|------|
| `soup.select("CSS선택자")` | 리스트 | 일치하는 모든 요소 |
| `soup.select_one("CSS선택자")` | 단일 요소 | 첫 번째 일치 요소 |
| `elem.text` | str | 태그 안의 텍스트 |
| `elem.get_text(strip=True)` | str | 텍스트 (공백 제거) |
| `elem.get("href")` | str | 속성값 가져오기 |
| `elem["href"]` | str | 속성값 가져오기 (없으면 에러) |

### 네이버 뉴스 API 요청 핵심 파라미터

| 파라미터 | 설명 | 예시 값 |
|----------|------|---------|
| `query` | 검색어 (URL 인코딩) | `%ED%8C%8C%EC%9D%B4%EC%8D%AC` |
| `sort` | 정렬 | `0`=관련도, `1`=최신, `2`=오래된순 |
| `ds` / `de` | 시작/종료 날짜 | `2025.01.01` / `2025.03.01` |
| `start` | 시작 위치 | `1`, `11`, `21`, ... |
| `news_office_checked` | 언론사 ID | `1056` (KBS) |
| `photo` | 뉴스 유형 | `0`=전체, `1`=포토, `2`=동영상 |

### 스크래핑 시 주의사항

| 항목 | 내용 |
|------|------|
| User-Agent 헤더 | 브라우저로 위장 필수 (차단 방지) |
| 요청 간격 | `time.sleep(1)` 이상 대기 |
| 클래스명 변경 | 네이버는 주기적으로 HTML 구조 변경 → 개발자 도구로 재확인 |
| 최대 수집량 | 400페이지 (4,000건) 초과 시 날짜 분할 수집 |